# Anomaly Detection

> In this notebook, we'll implement anomaly detection in time series data 

::: {.callout-warning collapse="true"}

## Prerequesites

This tutorial assumes basic familiarity with StatsForecast. For a minimal example visit the [Quick Start](https://nixtla.github.io/statsforecast/examples/getting_started_short.html) 
:::

## Introduction 

Anomaly detection is a crucial task in time series forecasting. It involves identifying unusual observations that don't follow the expected dataset patterns. Anomalies, also known as outliers, can be caused by a variety of factors, such as errors in the data collection process, sudden changes in the underlying patterns of the data, or unexpected events. They can pose problems for many forecasting models since they can distort trends, seasonal patterns, or autocorrelation estimates. As a result, anomalies can have a significant impact on the accuracy of the forecasts, and for this reason, it is essential to be able to identify them. Furthermore, anomaly detection has many applications across different industries, such as detecting fraud in financial data, monitoring the performance of online services, or identifying usual patterns in energy usage.

By the end of this tutorial, you'll have a good understanding of how to detect anomalies in time series data using [StatsForecast](https://nixtla.github.io/statsforecast/)'s probabilistic models. 

**Outline:**

1. Install libraries 
2. Load and explore data 
3. Train model 
4. Recover insample forecasts and identify anomalies

::: {.callout-important}

## Important 

Once an anomaly has been identified, we must decide what to do with it. For example, we could remove it or replace it with another value. The correct course of action is context-dependent and beyond this notebook's scope. Removing an anomaly will likely improve the accuracy of the forecast, but it can also underestimate the amount of randomness in the data.  
:::

:::{.callout-tip}
You can use Colab to run this Notebook interactively <a href="https://colab.research.google.com/github/Nixtla/statsforecast/blob/main/nbs/examples/AnomalyDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
:::

## Install libraries 

We assume that you have StatsForecast already installed. If not, check this guide for instructions on [how to install StatsForecast](https://nixtla.github.io/statsforecast/examples/installation.html) 

Install the necessary packages using `pip install statsforecast`

In [ ]:
pip install statsforecast -U

## Load and explore the data 

For this example, we'll use the hourly dataset of the [M4 Competition](https://www.sciencedirect.com/science/article/pii/S0169207019301128). We'll first import the data from [datasetsforecast](https://github.com/Nixtla/datasetsforecast/tree/main/), which you can install using `pip install datasetsforecast`

In [ ]:
pip install datasetsforecast -U 

In [ ]:
from datasetsforecast.m4 import M4 

The function to load the data is `M4.load`. It requieres the following two arguments: 

- `directory`: (str) The directory where the data will be downloaded. 
- `group`: (str). The group name, which can be `Yearly`, `Quarterly`, `Monthly`, `Weekly`, `Daily` or `Hourly`. 

This function returns multiple outputs, but only the first one with the target series is needed. 

In [ ]:
df_total, *_ = M4.load('./data', 'Hourly')
df_total.head()

The input to StatsForecast is always a data frame in [long format](https://www.theanalysisfactor.com/wide-and-long-data/) with three columns: `unique_id`, `df` and `y`.  

- `unique_id`: (string, int or category) A unique identifier for the series. 
- `ds`: (datestamp or int) A datestamp in format YYYY-MM-DD or YYYY-MM-DD HH:MM:SS or an integer indexing time. 
- `y`: (numeric) The measurement we wish to forecast. 

In this case, the `unique_id` and `y` columns already have the requiered format, but we need to change the data type of the `ds` column.  

In [ ]:
df_total['ds'] = df_total['ds'].astype(int)

From this dataset, we'll select the first 8 time series to reduce the total execution time. You can select any number you want by changing the value of `n_series`. 

In [ ]:
n_series = 8 
uids = df_total['unique_id'].unique()[:n_series]
df = df_total.query('unique_id in @uids')

We can plot these series using the `plot` method from the StatsForecast class. This method has multiple parameters, and the required ones to generate the plots in this notebook are explained below. 

- `df`: A pandas dataframe with columns [unique_id, ds, y].
- `forecasts_df`: A pandas dataframe with columns [unique_id, ds] and models.
- `unique_ids`: (list[str]) A list with the ids of the time series we want to plot. 
- `plot_random`: (bool = True) Plots the time series randomly.
- `plot_anomalies`: (bool = False) Plots anomalies for each prediction interval. 
- `engine`: (str = plotly). The library used to generate the plots. It can also be matplotlib for static plots.

In [ ]:
from statsforecast import StatsForecast 

In [ ]:
StatsForecast.plot(df, plot_random = False) 

## Train model 

To generate the forecast, we'll use the [MSTL](https://nixtla.github.io/statsforecast/models.html#multiple-seasonalities) model, which is well-suited for low-frequency data like the one used here. We first need to import it from `statsforecast.models` and then we need to instantiate it. Since we're using hourly data, we have two seasonal periods: one every 24 hours (hourly) and one every 24\*7 hours (daily). Hence, we need to set `season_length = [24, 24*7]`. 

In [ ]:
from statsforecast.models import MSTL

# Create a list of models and instantiation parameters 
models = [MSTL(season_length = [24, 24*7])]

To instantiate a new StatsForecast object, we need the following parameters:

- `df`: The dataframe with the training data.
- `models`: The list of models defined in the previous step.
- `freq`: A string indicating the frequency of the data. See [pandas’ available frequencies](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases). 
- `n_jobs`: An integer that indicates the number of jobs used in parallel processing. Use -1 to select all cores.

In [ ]:
sf = StatsForecast(
    df = df, 
    models = models, 
    freq = 'H', 
    n_jobs = -1
)

We'll now predict the next 48 hours. To do this, we'll use the `forecast` method, which requieres the following arguments: 

- `h`: (int) The forecasting horizon. 
- `level`: (list[float]) The confidence levels of the prediction intervals
- `fitted`: (bool = False) Returns insample predictions. 

It is important that we select a `level` and set `fitted = True` since we'll need the insample forecasts and their prediction intervals to detect the anomalies. 

In [ ]:
horizon = 48
levels = [99] 

fcst = sf.forecast(h = 48, level = levels, fitted = True)
fcst = fcst.reset_index()
fcst.head()

We can plot the forecasts using the `plot` method from before. 

In [ ]:
StatsForecast.plot(df, fcst, plot_random = False) 

## Recover insample forecasts and identify anomalies

In this example, an **anomaly** will be any observation outside the prediction interval of the insample forecasts for a given confidence level (here we selected 99%). Hence, we first need to recover the insample forecasts using the `forecast_fitted_values` method. 

In [ ]:
insample_forecasts = sf.forecast_fitted_values().reset_index()
insample_forecasts.head()

We can now find all the observations above or below the 99% prediction interval for the insample forecasts. 

In [ ]:
anomalies = insample_forecasts.loc[(insample_forecasts['y'] >= insample_forecasts['MSTL-hi-99']) | (insample_forecasts['y'] <= insample_forecasts['MSTL-lo-99'])]
anomalies.head()

We can plot the anomalies by adding the `plot_anomalies = True` argument to the `plot` method. 

In [ ]:
StatsForecast.plot(insample_forecasts, plot_random = False, plot_anomalies = True)

If we want to take a closer look, we can use the `unique_ids` argument to select one particular time series, for example, `H10`. 

In [ ]:
StatsForecast.plot(insample_forecasts, unique_ids = ['H10'], plot_anomalies = True) 

Here we identified the anomalies in the data using the MSTL model, but any [probabilistic model](https://nixtla.github.io/statsforecast/#models) from StatsForecast can be used. We also selected the 99% prediction interval of the insample forecasts, but other confidence levels can be used as well. 